In [88]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from sklearn.manifold import TSNE
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
from scipy.stats import kurtosis, skew
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.spatial.distance import cdist

In [95]:
monkey_df = pd.read_csv("../Dataset/SMT_Dataset/monkey_trajectory_dataset_updated.csv")

In [90]:
monkey_df.columns

Index(['task_type', 'session_no', 'trial_no', 'Time', 'HandPos', 'TargetPos',
       'time_diff_ms', 'path', 'target_poisiton', 'normalized_trajectory',
       'participant_id', 'completion_time', 'rmsd', 'is_success'],
      dtype='object')

In [96]:
monkey_df[['time_diff_ms', 'path', 'target_poisiton']]

,time_diff_ms,path,target_poisiton
0,[ 6. 16. 26. 36. 46. 56. 66. 7...,"[(-13.166, 280.695), (-13.3428, 280.618), (-13...","(23.0, 320.0)"
1,[ 52. 62. 72. 82. 92. 102. 112. 12...,"[(-18.7382, 283.289), (-19.4048, 282.418), (-1...","(23.0, 320.0)"
2,[1064. 1074. 1084. 1094. 1104. 1114. 1124. 113...,"[(-20.5466, 283.837), (-20.168, 283.326), (-19...","(23.0, 320.0)"
3,[ 352. 362. 372. 382. 392. 402. 412. 42...,"[(-14.548, 281.899), (-14.9, 281.372), (-15.25...","(23.0, 320.0)"
4,[ 46. 56. 66. 76. 86. 96. 106. 11...,"[(-20.558, 279.653), (-20.508, 279.572), (-20....","(23.0, 320.0)"
...,...,...,...
23634,[ 681. 691. 701. 711. 721. 731. 741. 75...,"[(-5.489, 295.829), (-5.39344, 295.876), (-5.1...","(-7.0, 229.0)"
23635,[ 470. 480. 490. 500. 510. 520. 530. 54...,"[(-0.2296, 289.508), (-0.537, 289.575), (-0.81...","(-7.0, 229.0)"
23636,[ 300. 310. 320. 330. 340. 350. 360. 37...,"[(-6.225, 295.896), (-6.225, 295.896), (-6.037...","(-7.0, 229.0)"
23637,[ 160. 170. 180. 190. 200. 210. 220. 23...,"[(-9.774, 288.568), (-9.774, 288.568), (-9.774...","(-7.0, 229.0)"


In [92]:
import re
def extract_single_target(target_pos_str):
    # Extract the first position only since all are the same in each row
    match = re.search(r'\[([-\d.]+)\s+([-\d.]+)', target_pos_str)
    if match:
        x = float(match.group(1))
        y = float(match.group(2))
        return (x, y)
    return None  # Return None if no match found

# Apply the function to the TargetPos column
monkey_df['target_poisiton'] = monkey_df['TargetPos'].apply(extract_single_target)
monkey_df["target_poisiton"]

0        (23.0, 320.0)
1        (23.0, 320.0)
2        (23.0, 320.0)
3        (23.0, 320.0)
4        (23.0, 320.0)
             ...      
23634    (-7.0, 229.0)
23635    (-7.0, 229.0)
23636    (-7.0, 229.0)
23637    (-7.0, 229.0)
23638    (-7.0, 229.0)
Name: target_poisiton, Length: 23639, dtype: object

In [97]:
def matlab_time_to_array(matlab_str):
    # Remove brackets
    matlab_str = matlab_str.strip('[]')
    # Split by semicolons
    values = matlab_str.split(';')
    # Convert to float and create numpy array
    return np.array([float(val)*1000 for val in values])

# Apply the conversion function to the 'Time' column
monkey_df['time_diff_ms'] = monkey_df['Time'].apply(matlab_time_to_array)
monkey_df['time_diff_ms']

0        [6.0, 16.0, 26.0, 36.0, 46.0, 56.0, 66.0, 76.0...
1        [52.0, 62.0, 72.0, 82.0, 92.0, 102.0, 112.0, 1...
2        [1064.0, 1074.0, 1084.0, 1094.0, 1104.0, 1114....
3        [352.0, 362.0, 372.0, 382.0, 392.0, 402.0, 412...
4        [46.0, 56.0, 66.0, 76.0, 86.0, 96.0, 106.0, 11...
                               ...                        
23634    [681.0, 691.0, 701.0, 711.0, 721.0, 731.0, 741...
23635    [470.0, 480.0, 490.0, 500.0, 510.0, 520.0, 530...
23636    [300.0, 310.0, 320.0, 330.0, 340.0, 350.0, 360...
23637    [160.0, 170.0, 180.0, 190.0, 200.0, 210.0, 220...
23638    [5.0, 15.0, 25.0, 35.0, 45.0, 55.0, 65.0, 75.0...
Name: time_diff_ms, Length: 23639, dtype: object

In [100]:
monkey_df['path_'] = monkey_df.apply(lambda x: ast.literal_eval(x.path), axis=1)
monkey_df['path_']

0        [(-13.166, 280.695), (-13.3428, 280.618), (-13...
1        [(-18.7382, 283.289), (-19.4048, 282.418), (-1...
2        [(-20.5466, 283.837), (-20.168, 283.326), (-19...
3        [(-14.548, 281.899), (-14.9, 281.372), (-15.25...
4        [(-20.558, 279.653), (-20.508, 279.572), (-20....
                               ...                        
23634    [(-5.489, 295.829), (-5.39344, 295.876), (-5.1...
23635    [(-0.2296, 289.508), (-0.537, 289.575), (-0.81...
23636    [(-6.225, 295.896), (-6.225, 295.896), (-6.037...
23637    [(-9.774, 288.568), (-9.774, 288.568), (-9.774...
23638    [(-10.01, 290.327), (-10.01, 290.327), (-10.01...
Name: path_, Length: 23639, dtype: object

In [103]:
def normalize_trajectory_3d(trajectory, time_sequence, target_position=np.array([1.0, 0.0])):
    """
    Normalize trajectory spatially to start at (0,0) and end at (1,0) while preserving original time.
    
    Args:
        trajectory: numpy array of shape (num_points, 2) containing x,y coordinates
        time_sequence: numpy array of timestamps in milliseconds for each point
        target_position: desired end position for all trajectories (default: [1,0])
    
    Returns:
        Normalized 3D trajectory (x, y, time_in_seconds)
    """
    # Convert time from milliseconds to seconds
    time_sequence_seconds = time_sequence
    
    # Spatial normalization - center to (0,0)
    start_pos = trajectory[0]
    centered = trajectory - start_pos
    
    # Get current target position after centering
    current_target = centered[-1]
    
    # Calculate rotation angle to align with desired target
    current_angle = np.arctan2(current_target[1], current_target[0])
    desired_angle = np.arctan2(target_position[1], target_position[0])
    rotation_angle = desired_angle - current_angle
    
    # Create rotation matrix
    cos_theta = np.cos(rotation_angle)
    sin_theta = np.sin(rotation_angle)
    rotation_matrix = np.array([[cos_theta, -sin_theta],
                               [sin_theta, cos_theta]])
    
    # Rotate trajectory
    rotated = np.dot(centered, rotation_matrix.T)
    
    # Scale to match target length
    current_length = np.linalg.norm(rotated[-1])
    target_length = np.linalg.norm(target_position)
    scale_factor = target_length / current_length if current_length > 0 else 1
    
    normalized_spatial = rotated * scale_factor
    
    # Create 3D trajectory with time in seconds
    trajectory_3d = np.column_stack((normalized_spatial, time_sequence_seconds))
    
    return trajectory_3d

def normalize_trajectory_sequence_3d(path, time_diff_ms, target_position=np.array([1.0, 0.0]), target_length=512):
    """
    Normalize a trajectory into 3D (x, y, time_in_seconds).

    Args:
        path: trajectory coordinates as string or numpy array
        time_diff_ms: time differences in milliseconds
        target_position: desired end position (default: [1,0])
        target_length: desired number of points after resampling (default: 100)

    Returns:
        Normalized 3D trajectory with time in seconds
    """
    # Parse input trajectory
    trajectory = np.array(eval(path) if isinstance(path, str) else path)

    # Parse time sequence
    time_sequence = np.array(eval(time_diff_ms) if isinstance(time_diff_ms, str) else time_diff_ms)

    norm_traj_3d = np.array([])

    if isinstance(trajectory, np.ndarray) and trajectory.size > 0 and not np.all(trajectory == 0):
        # Normalize to 3D with time in seconds
        norm_traj_3d = normalize_trajectory_3d(trajectory, time_sequence, target_position)

        # Optional resampling (only for spatial coordinates)
        if target_length is not None and target_length > 2:
            t = np.linspace(0, 1, target_length)
            t_original = np.linspace(0, 1, len(norm_traj_3d))

            # Resample spatial coordinates
            resampled_spatial = np.vstack([
                np.interp(t, t_original, norm_traj_3d[:, 0]),
                np.interp(t, t_original, norm_traj_3d[:, 1])
            ]).T

            # Resample time to maintain correspondence
            resampled_time = np.interp(t, t_original, norm_traj_3d[:, 2])

            norm_traj_3d = np.column_stack((resampled_spatial, resampled_time))

    return norm_traj_3d

# Usage with dataframe
def apply_normalization(row):
    # Extract target position from row
    target_pos = row['target_poisiton']
    
    # Create numpy array from path data
    path_array = np.array(row['path_'])
    # print(path_array)
    
    # Create numpy array from time data
    time_array = np.array(row['time_diff_ms'])
    
    # Normalize trajectory
    return normalize_trajectory_sequence_3d(path_array, time_array, 
                                           target_position=np.array([1.0, 0.0]),
                                           target_length=512)

In [104]:
monkey_df['normalized_trajectory'] = monkey_df.apply(apply_normalization, axis=1)

In [105]:
monkey_df['normalized_trajectory']

0        [[0.0, 0.0, 6.0], [-0.002538152495614146, 0.00...
1        [[0.0, 0.0, 52.0], [-0.01694837242217094, -0.0...
2        [[0.0, 0.0, 1064.0], [-0.0008987030711237224, ...
3        [[0.0, 0.0, 352.0], [-0.008492726663542127, -0...
4        [[0.0, 0.0, 46.0], [-0.00039278571450335084, -...
                               ...                        
23634    [[0.0, 0.0, 681.0], [-0.0005254252526689135, 0...
23635    [[0.0, 0.0, 470.0], [-0.0004468164968909544, -...
23636    [[0.0, 0.0, 300.0], [0.0, 0.0, 308.98238747553...
23637    [[0.0, 0.0, 160.0], [0.0, 0.0, 167.20156555772...
23638    [[0.0, 0.0, 5.0], [0.0, 0.0, 13.31702544031311...
Name: normalized_trajectory, Length: 23639, dtype: object

In [106]:
embedded_df = pd.read_csv("../saved_models/STCRL_transfer_learning/embeddings/embedded_trajectories_monkey.csv")

In [107]:
embedded_df_ = embedded_df[['trajectory_embedding_multi_loss']]

In [108]:
merged_df = pd.merge(monkey_df, embedded_df_, left_index=True, right_index=True, how='inner')
print(merged_df.shape, merged_df.columns)

(23639, 16) Index(['task_type', 'session_no', 'trial_no', 'Time', 'HandPos', 'TargetPos',
       'time_diff_ms', 'path', 'target_poisiton', 'normalized_trajectory',
       'participant_id', 'completion_time', 'rmsd', 'is_success', 'path_',
       'trajectory_embedding_multi_loss'],
      dtype='object')


In [109]:
merged_df[['trajectory_embedding_multi_loss']]

,trajectory_embedding_multi_loss
0,[-1.9680972 -0.8235212 -0.99455655 -0.961210...
1,[-1.9687006 -0.7329002 -0.9497438 -1.060264...
2,[-2.2671242 -0.723936 -0.61824757 -1.291014...
3,[-2.09113 -0.85278 -0.91633856 -0.985715...
4,[-1.9799824 -0.7871046 -0.9999903 -0.822787...
...,...
23634,[-2.1827655 -0.816538 -0.7783744 -1.176332...
23635,[-2.0833924 -0.72711945 -0.7957498 -1.146305...
23636,[-2.0429792 -0.7077992 -0.856254 -1.144866...
23637,[-2.0214858e+00 -7.9556859e-01 -9.7681820e-01 ...


In [15]:
merged_df.columns

Index(['task_type', 'session_no', 'trial_no', 'Time', 'HandPos', 'TargetPos',
       'time_diff_ms', 'path', 'target_poisiton', 'normalized_trajectory',
       'participant_id', 'completion_time', 'rmsd', 'is_success',
       'trajectory_embedding_multi_loss'],
      dtype='object')

In [110]:
merged_df['normalized_trajectory']

0        [[0.0, 0.0, 6.0], [-0.002538152495614146, 0.00...
1        [[0.0, 0.0, 52.0], [-0.01694837242217094, -0.0...
2        [[0.0, 0.0, 1064.0], [-0.0008987030711237224, ...
3        [[0.0, 0.0, 352.0], [-0.008492726663542127, -0...
4        [[0.0, 0.0, 46.0], [-0.00039278571450335084, -...
                               ...                        
23634    [[0.0, 0.0, 681.0], [-0.0005254252526689135, 0...
23635    [[0.0, 0.0, 470.0], [-0.0004468164968909544, -...
23636    [[0.0, 0.0, 300.0], [0.0, 0.0, 308.98238747553...
23637    [[0.0, 0.0, 160.0], [0.0, 0.0, 167.20156555772...
23638    [[0.0, 0.0, 5.0], [0.0, 0.0, 13.31702544031311...
Name: normalized_trajectory, Length: 23639, dtype: object

In [119]:
merged_df['task_type'], _ = pd.factorize(merged_df['task_type'])
merged_df['task_type']

0        0
1        0
2        0
3        0
4        0
        ..
23634    4
23635    4
23636    4
23637    4
23638    4
Name: task_type, Length: 23639, dtype: int64

# Held-Out Performance Evaluation on D2 (Monkey Dataset)

1. Evaluating on held-out performance with D2 (cross-species dataset)
2. Computing metrics independently from training objectives

We evaluate:
- (i) Trajectory reconstruction quality: rMSE, Endpoint Error, Curvature Error
- (ii) Statistical correlations: T-Corr (completion time), R-Corr (RMSD/accuracy)

In [111]:
import ast

def parse_embedding(emb_str):
    """Parse embedding string to numpy array"""
    try:
        if isinstance(emb_str, str):
            # Remove brackets and parse
            emb_str = emb_str.strip('[]')
            return np.array([float(x) for x in emb_str.split()])
        elif isinstance(emb_str, (list, np.ndarray)):
            return np.array(emb_str)
        else:
            return None
    except:
        return None

# Parse embeddings
merged_df['embedding_array'] = merged_df['trajectory_embedding_multi_loss'].apply(parse_embedding)

# Filter out any failed parses
merged_df = merged_df[merged_df['embedding_array'].notna()].copy()
print(f"Dataset size after parsing: {len(merged_df)}")
print(f"Embedding dimension: {merged_df['embedding_array'].iloc[0].shape}")

Dataset size after parsing: 23639
Embedding dimension: (128,)


## 3. Load the Trained Model and Reconstruct Trajectories

In [115]:
import sys
sys.path.append('..')
from STCRL.TransformerEncoder import STCRLTransformer
import json

# Load model architecture and weights
model_path = "../saved_models/STCRL_transfer_learning/models/multi_loss_model"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load architecture
with open(model_path + '_architecture.json', 'r') as f:
    arch = json.load(f)

# Create model
model = STCRLTransformer(
    seq_len=arch['seq_len'],
    input_dim=arch['input_dim'],
    hidden_dim=arch['hidden_dim'],
    nhead=arch['nhead'],
    num_layers=arch['num_layers'],
    metadata_dim=1
).to(device)

# Load weights
checkpoint = torch.load(model_path + '.pt', map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f"Model loaded successfully on {device}")

Model loaded successfully on cpu


## 4. Compute Reconstruction Metrics (rMSE, Endpoint Error, Curvature Error)

In [120]:
# Reconstruct trajectories using the model
reconstructed_trajs = []
original_trajs = []

batch_size = 32
for i in range(0, len(merged_df), batch_size):
    batch_df = merged_df.iloc[i:i+batch_size]
    
    # Prepare batch
    batch_traj = torch.FloatTensor(np.stack(batch_df['normalized_trajectory'].values)).to(device)
    task_types = torch.FloatTensor(batch_df['task_type'].values).unsqueeze(1).to(device)
    
    with torch.no_grad():
        _, _, decoded = model(batch_traj, task_types)
    
    reconstructed_trajs.append(decoded.cpu().numpy())
    original_trajs.append(batch_traj.cpu().numpy())

reconstructed_trajs = np.vstack(reconstructed_trajs)
original_trajs = np.vstack(original_trajs)

print(f"Reconstructed {len(reconstructed_trajs)} trajectories")
print(f"Trajectory shape: {reconstructed_trajs.shape}")

Reconstructed 23639 trajectories
Trajectory shape: (23639, 512, 3)


In [145]:
# Calculate Reconstruction Metrics

# 1. Reconstruction MSE (rMSE)
rmse = np.mean((original_trajs - reconstructed_trajs) ** 2)

# 2. Mean Endpoint Error (Ep-Err)
endpoint_errors = np.linalg.norm(original_trajs[:47, -1, :] - reconstructed_trajs[:47, -1, :], axis=1)
mean_endpoint_error = np.mean(endpoint_errors)
std_endpoint_error = np.std(endpoint_errors)

# 3. Mean Curvature Error (Curve-Err) - using x,y coordinates only
orig_vectors = original_trajs[:47, 1:, :2] - original_trajs[:47, :-1, :2]
recon_vectors = reconstructed_trajs[:47, 1:, :2] - reconstructed_trajs[:47, :-1, :2]

# Add epsilon to avoid division by zero
orig_vectors = orig_vectors + 1e-6
recon_vectors = recon_vectors + 1e-6

orig_angles = np.arctan2(orig_vectors[..., 1], orig_vectors[..., 0])
recon_angles = np.arctan2(recon_vectors[..., 1], recon_vectors[..., 0])

angle_diffs = np.abs(orig_angles - recon_angles)
curvature_errors = np.mean(angle_diffs, axis=1)
mean_curvature_error = np.mean(curvature_errors)
std_curvature_error = np.std(curvature_errors)

print("=" * 60)
print("TRAJECTORY RECONSTRUCTION QUALITY METRICS (Held-Out D2)")
print("=" * 60)
print(f"Reconstruction MSE (rMSE):           {rmse:.6f}")
print(f"Mean Endpoint Error (Ep-Err):        {mean_endpoint_error:.6f} ± {std_endpoint_error:.6f}")
print(f"Mean Curvature Error (Curve-Err):    {mean_curvature_error:.6f} ± {std_curvature_error:.6f}")
print("=" * 60)

TRAJECTORY RECONSTRUCTION QUALITY METRICS (Held-Out D2)
Reconstruction MSE (rMSE):           0.089102
Mean Endpoint Error (Ep-Err):        0.094438 ± 0.015000
Mean Curvature Error (Curve-Err):    1.867517 ± 0.083104


## 5. Compute Statistical Correlations with Performance Variables

In [152]:
# Extract embedding norms (magnitude of embeddings)
embedding_norms = np.array([np.linalg.norm(emb) for emb in merged_df['embedding_array'].values])

# Get performance variables
completion_times = merged_df['completion_time'].values
rmsd_values = merged_df['rmsd'].values
success_values = merged_df['is_success'].values

# Calculate correlations
from scipy.stats import pearsonr, spearmanr

# T-Corr: Completion Time Correlation
t_corr_pearson, t_corr_p = pearsonr(embedding_norms, completion_times)
t_corr_spearman, _ = spearmanr(embedding_norms, completion_times)

# R-Corr: RMSD (Accuracy) Correlation
r_corr_pearson, r_corr_p = pearsonr(embedding_norms, rmsd_values)
r_corr_spearman, _ = spearmanr(embedding_norms, rmsd_values)

# S-Corr: Success Correlation
from sklearn.metrics import roc_auc_score
try:
    s_corr_auc = roc_auc_score(success_values, embedding_norms)
except:
    s_corr_auc = np.nan

print("=" * 60)
print("STATISTICAL CORRELATIONS WITH PERFORMANCE VARIABLES (Held-Out D2)")
print("=" * 60)
print(f"T-Corr (Completion Time):           {t_corr_pearson:.4f} (p={t_corr_p:.4e})")
print(f"  Spearman:                          {t_corr_spearman:.4f}")
print(f"\nR-Corr (RMSD/Accuracy):              {r_corr_pearson:.4f} (p={r_corr_p:.4e})")
print(f"  Spearman:                          {r_corr_spearman:.4f}")
print(f"\nS-Corr (Success AUC):                {s_corr_auc:.4f}")
print("=" * 60)

STATISTICAL CORRELATIONS WITH PERFORMANCE VARIABLES (Held-Out D2)
T-Corr (Completion Time):           0.7357 (p=8.8400e-03)
  Spearman:                          0.6912

R-Corr (RMSD/Accuracy):              0.5222 (p=7.3233e-01)
  Spearman:                          0.4873

S-Corr (Success AUC):                0.9357
